# Aeromant — CFD with OpenFOAM template cases

Aeromant is standalone: it takes any STL, a known-good template case and explicit values.
Here the STL is a sphere generated with numpy so the notebook does not need a CAD package.
Every step (prepare, mesh, solve) is started explicitly.

In [ ]:
import math
from pathlib import Path
import numpy as np
from vegeta import aeromant

RUNS = Path("_runs/aeromant"); RUNS.mkdir(parents=True, exist_ok=True)

## 1. Geometry: an STL from anywhere (units must be stated explicitly later)

In [ ]:
def icosphere(radius, level):
    t = (1 + 5 ** 0.5) / 2
    v = [(-1, t, 0), (1, t, 0), (-1, -t, 0), (1, -t, 0), (0, -1, t), (0, 1, t), (0, -1, -t), (0, 1, -t),
         (t, 0, -1), (t, 0, 1), (-t, 0, -1), (-t, 0, 1)]
    f = [(0, 11, 5), (0, 5, 1), (0, 1, 7), (0, 7, 10), (0, 10, 11), (1, 5, 9), (5, 11, 4), (11, 10, 2), (10, 7, 6),
         (7, 1, 8), (3, 9, 4), (3, 4, 2), (3, 2, 6), (3, 6, 8), (3, 8, 9), (4, 9, 5), (2, 4, 11), (6, 2, 10),
         (8, 6, 7), (9, 8, 1)]
    verts = [np.array(p, float) / np.linalg.norm(p) for p in v]
    for _ in range(level):
        cache, nf = {}, []
        def mid(a, b):
            key = (min(a, b), max(a, b))
            if key not in cache:
                m = verts[a] + verts[b]; verts.append(m / np.linalg.norm(m)); cache[key] = len(verts) - 1
            return cache[key]
        for a, b, c in f:
            ab, bc, ca = mid(a, b), mid(b, c), mid(c, a)
            nf += [(a, ab, ca), (b, bc, ab), (c, ca, bc), (ab, bc, ca)]
        f = nf
    return aeromant.Surface((np.array(verts) * radius)[np.array(f)], "sphere")

stl = aeromant.write_stl_ascii(icosphere(0.5, 4), RUNS / "sphere.stl")

## 2. Choose a template and read what it decides and what it needs

In [ ]:
print(aeromant.get_template('laminar_external').describe())

## 3. Tell Aeromant how to launch OpenFOAM
`detect()` finds the OpenFOAM that `install_local.sh` installed (conda env in `/opt/foam`), official
openfoam.com/.org packages, or an already sourced environment. You can also be explicit:
`OpenFOAMEnvironment(bashrc=".../etc/bashrc")` or `OpenFOAMEnvironment.conda("/opt/foam")`.

In [ ]:
env = aeromant.OpenFOAMEnvironment.detect()
env

## 4. Define the case with explicit values and prepare it

In [ ]:
case = aeromant.CFDCase(
    "laminar_external", stl,
    dict(velocity=1.0, kinematic_viscosity=0.01, density=1.0,          # Re = U D / nu = 100
         reference_area=math.pi / 4, reference_length=1.0, center_of_rotation=(0, 0, 0)),
    workdir=RUNS / "sphere_re100", geometry_units="m", environment=env,
)
case.prepare(overwrite=True)

## 5. Mesh only — inspect before solving

In [ ]:
case.run(steps=['blockMesh', 'features', 'snappyHexMesh', 'checkMesh'])

## 6. Solve (explicitly)

In [ ]:
res = case.run(steps=['restore0', 'solver'], progress=True)
res

In [ ]:
if res.ok:
    print("Aeromant Cd:            ", res.metrics["Cd"])
    print("Schiller-Naumann Cd(100):", 24 / 100 * (1 + 0.15 * 100 ** 0.687))
    fig1 = aeromant.plot_coefficients(case.workdir)
    fig2 = aeromant.plot_residuals(case.workdir)
else:
    print("The solver run failed — see the messages above and the log:", res.artifacts.get("log_solver"))

## 7. Results can be re-read at any time without running anything

In [ ]:
again = aeromant.read_case_results(case.workdir)
again.metrics.get("Cd_mean_last50") if again.ok else again.messages